# Phantom Ground-Truth Validation

This notebook is a thin, reproducible wrapper around `scripts/validate_phantom_tracking.py`. It is deliberately reviewer-style: if ground truth or tracker output is missing, it reports that explicitly instead of producing reassuring plots without validation value.

## Coordinate and unit contract

- Strict runner line segments are stored as MATLAB-style one-based pixel coordinates `[x1, y1, x2, y2]`.
- Displacement is computed from segment midpoint relative to frame 0, so the one-based coordinate offset cancels.
- Image `x` is lateral/rightward; image `y` is axial/downward.
- The current `mm_per_pixel` is a scalar depth/height conversion. It is valid for axial displacement if the image depth metadata is correct; lateral/vector validation needs a lateral spacing calibration unless square pixels are known.
- `raw` metrics mean no frame lag and no sign flip. `best_aligned` metrics are diagnostic only and should not be used to hide a synchronization or sign-convention error.
- For `june29_3`, the phantom setup is expected to move left-right, so lateral `x` displacement is the primary validation axis and axial `y` displacement is treated as bounce/orthogonal-motion.
- In this run the imposed displacement is entered as a 15 mm cumulative lateral ramp. This is an assumption about timing: linear from first to last frame unless replaced by an actuator trace.

In [1]:
from pathlib import Path
import os
import subprocess
import pandas as pd

PROJECT = Path.cwd().resolve()
if not (PROJECT / 'ultrasound_tracker').exists():
    PROJECT = PROJECT.parent.resolve()

VIDEO = PROJECT / 'data' / 'raw' / 'june29_3.mp4'
# Set this to the real actuator/phantom displacement file when available.
GROUND_TRUTH = None
# Known imposed plate/probe travel. Without an actuator trace, validate against a linear 0->15 mm lateral ramp.
TOTAL_X_MM = 15.0
OUT = PROJECT / 'results' / 'phantom_ground_truth_validation' / 'june29_3'

VIDEO, GROUND_TRUTH, TOTAL_X_MM, OUT

(PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/data/raw/june29_3.mp4'),
 None,
 15.0,
 PosixPath('/Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3'))

In [2]:
cmd = [
    str(PROJECT / '.venv' / 'bin' / 'python'),
    str(PROJECT / 'scripts' / 'validate_phantom_tracking.py'),
    '--video', str(VIDEO),
    '--out-dir', str(OUT),
    '--axis', 'x',
]
if GROUND_TRUTH is not None:
    cmd += ['--ground-truth', str(GROUND_TRUTH)]
elif TOTAL_X_MM is not None:
    cmd += ['--synthetic-total-x-mm', str(TOTAL_X_MM)]

env = dict(os.environ)
env.setdefault('MPLCONFIGDIR', '/private/tmp/matplotlib')
print(' '.join(cmd))
subprocess.run(cmd, cwd=str(PROJECT), env=env, check=True)

/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python /Users/grosbedou/PycharmProjects/NDORMS/scripts/validate_phantom_tracking.py --video /Users/grosbedou/PycharmProjects/NDORMS/data/raw/june29_3.mp4 --out-dir /Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3 --axis x --synthetic-total-x-mm 15.0


Report: /Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3/phantom_validation_report.md
Summary CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3/phantom_validation_summary.csv
Per-frame CSV: /Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3/phantom_validation_per_frame.csv


CompletedProcess(args=['/Users/grosbedou/PycharmProjects/NDORMS/.venv/bin/python', '/Users/grosbedou/PycharmProjects/NDORMS/scripts/validate_phantom_tracking.py', '--video', '/Users/grosbedou/PycharmProjects/NDORMS/data/raw/june29_3.mp4', '--out-dir', '/Users/grosbedou/PycharmProjects/NDORMS/results/phantom_ground_truth_validation/june29_3', '--axis', 'x', '--synthetic-total-x-mm', '15.0'], returncode=0)

In [3]:
report_path = OUT / 'phantom_validation_report.md'
print(report_path.read_text())

# Phantom Ground-Truth Validation

## Data Audit
- Video: `/Users/grosbedou/PycharmProjects/NDORMS/data/raw/june29_3.mp4`
- Video opened: `True`
- Frames/FPS/size: `642` frames, `25` fps, `1024x768` px
- Strict runner NPZ: `/Users/grosbedou/PycharmProjects/NDORMS/results/strict_ultratimtrack_runs/june29_3/june29_3_strict_results.npz`
- Estimate sources: `final_kalman_midpoint, final_kalman_end_midpoint, klt_prior_midpoint, fixed_kalman_midpoint, klt_tracker_median_cumulative`
- mm_per_pixel: `0.06601562500000001`
- Ground truth: `synthetic_linear_cumulative_x15mm`
- GT columns used: `{'frame': 'video_frame', 'time_s': 'frame/fps', 'x': 'synthetic_total_x_mm', 'y': None, 'scalar': None}`
- GT axis type: `x`
- GT source note: synthetic linear cumulative ramp from the supplied total displacement; replace with an actuator/encoder trace if the plate motion was not constant-speed over the full video.
- Failure threshold: `0.5` mm

## Coordinate And Unit Contract
- Image origin is the top-lef

In [4]:
summary_path = OUT / 'phantom_validation_summary.csv'
summary = pd.read_csv(summary_path) if summary_path.exists() else pd.DataFrame()
summary

,source,comparison,axis,lag_frames,estimate_sign,n,mae_mm,rmse_mm,bias_mm,error_sd_mm,max_abs_error_mm,r2,pearson_r,endpoint_error_mm,drift_slope_mm_per_s,failure_rate
0,final_kalman_midpoint,raw,x/lateral,0,1.0,642,15.716827,17.159824,15.715863,6.895308,27.368565,-14.655663,0.988945,24.561098,0.902269,0.950156
1,final_kalman_midpoint,best_aligned,x/lateral,-10,1.0,632,15.349283,16.802880,15.341873,6.858436,27.134555,-14.489920,0.989021,24.517235,0.912035,0.949367
2,final_kalman_end_midpoint,raw,x/lateral,0,1.0,642,15.241686,16.631199,15.240642,6.662487,26.286748,-13.705943,0.986699,23.627099,0.865650,0.950156
3,final_kalman_end_midpoint,best_aligned,x/lateral,-10,1.0,632,14.881461,16.282627,14.873939,6.630192,26.052739,-13.545568,0.986722,23.540342,0.875407,0.949367
4,klt_prior_midpoint,raw,x/lateral,0,1.0,642,2.244434,2.999889,-2.164612,2.078586,6.526685,0.521529,0.963697,-6.526685,-0.265851,0.699377
5,klt_prior_midpoint,best_aligned,x/lateral,10,1.0,632,2.132346,2.857939,-1.963685,2.078121,6.292675,0.551887,0.963233,-6.292675,-0.270694,0.689873
6,fixed_kalman_midpoint,raw,x/lateral,0,1.0,642,17.598512,19.180594,17.597625,7.636071,30.126891,-18.560046,0.987512,27.217368,0.997808,0.950156
7,fixed_kalman_midpoint,best_aligned,x/lateral,-10,1.0,632,17.218295,18.811947,17.211500,7.599014,29.892882,-18.415529,0.987649,27.175307,1.009257,0.949367
8,klt_tracker_median_cumulative,raw,x/lateral,0,1.0,642,2.076618,2.756449,-1.978587,1.920665,6.020101,0.596033,0.970183,-6.020101,-0.244511,0.700935
9,klt_tracker_median_cumulative,best_aligned,x/lateral,10,1.0,632,1.964365,2.613957,-1.774699,1.920690,5.786092,0.625132,0.969802,-5.786092,-0.249039,0.683544
